In [6]:
import glob
import numpy as np
from astropy.io import fits

def load_and_stack(file_list):
    """Carica i file FITS e li impila in un array 3D numpy."""
    data_list = []
    for fname in file_list:
        with fits.open(fname) as hdul:
            data_list.append(hdul[0].data.astype(np.float64))
    return np.array(data_list)

def calculate_median_and_error(stack):
    """
    Calcola la mediana lungo l'asse 0 e propaga l'errore sulla mediana.
    Errore_mediana = 1.253 * std / sqrt(N)
    """
    n_frames = stack.shape[0]
    median_img = np.median(stack, axis=0)
    std_img = np.std(stack, axis=0)
    
    # Formula di propagazione dell'errore per la mediana
    error_img = (1.253 * std_img) / np.sqrt(n_frames)
    return median_img, error_img
    
def save_fits(filename, data, header):
    hdu = fits.PrimaryHDU(data, header=header)
    hdu.writeto(filename, overwrite=True)

# ---------------------------------------------------------
# 1. MASTER BIAS
# ---------------------------------------------------------
print("Processing Bias frames...")
bias_files = glob.glob("C:/Users/vinc_/OneDrive/Desktop/Università/Laurea magistrale/Laboratorio tecnologie astronomiche/calib/*bias.fit")
if not bias_files:
    raise FileNotFoundError("Nessun file Bias trovato.")

bias_stack = load_and_stack(bias_files)
master_bias, master_bias_err = calculate_median_and_error(bias_stack)

dark_header = fits.getheader(dark_files[0])
save_fits("master_dark.fits", master_dark_raw, dark_header)

save_fits("master_bias_err.fits", master_bias_err, None)
print(f"Creato Master Bias con mappa d'errore da {len(bias_files)} frame.")

# ---------------------------------------------------------
# 2. MASTER DARK (Sottrazione Bias e Propagazione Errore)
# ---------------------------------------------------------
print("\nProcessing Dark frames...")
dark_files = glob.glob("C:/Users/vinc_/OneDrive/Desktop/Università/Laurea magistrale/Laboratorio tecnologie astronomiche/calib/dark*.fit")
if not dark_files:
    raise FileNotFoundError("Nessun file Dark trovato.")

dark_stack = load_and_stack(dark_files)

# Sottraiamo il Bias da ogni singolo frame
for i in range(len(dark_stack)):
    dark_stack[i] -= master_bias

# Calcoliamo la mediana dei Dark sottratti e il loro errore intrinseco
master_dark_raw, dark_err_raw = calculate_median_and_error(dark_stack)

# Propagazione dell'errore (quadratura): sigma_dark_tot = sqrt(sigma_dark_raw^2 + sigma_bias^2)
master_dark_err = np.sqrt(dark_err_raw**2 + master_bias_err**2)

save_fits("master_dark.fits", master_dark_raw, None)
save_fits("master_dark_err.fits", master_dark_err, None)
print(f"Creato Master Dark con mappa d'errore da {len(dark_files)} frame.")

# ---------------------------------------------------------
# 3. MASTER FLAT (Sottrazione Bias+Dark, Normalizzazione ed Errore)
# ---------------------------------------------------------
print("\nProcessing Flat frames...")
flat_files = glob.glob("C:/Users/vinc_/OneDrive/Desktop/Università/Laurea magistrale/Laboratorio tecnologie astronomiche/calib/*flat.fit")
if not flat_files:
    raise FileNotFoundError("Nessun file Flat trovato.")

flat_stack = load_and_stack(flat_files)

# Sottraiamo Bias e Dark da ogni Flat
for i in range(len(flat_stack)):
    flat_stack[i] = flat_stack[i] - master_bias - master_dark_raw

# Mediana dei Flat e relativo errore intrinseco
master_flat_raw, flat_err_raw = calculate_median_and_error(flat_stack)

# Propagazione dell'errore della sottrazione: sigma_flat_sub = sqrt(sigma_flat^2 + sigma_bias^2 + sigma_dark^2)
flat_err_sub = np.sqrt(flat_err_raw**2 + master_bias_err**2 + master_dark_err**2)

#Normalizzazione
flat_median_value = np.median(master_flat_raw)
master_flat_normalized = master_flat_raw / flat_median_value
master_flat_err_normalized = flat_err_sub / flat_median_value

save_fits("master_flat.fits", master_flat_normalized, None)
save_fits("master_flat_err.fits", master_flat_err_normalized, None)
print(f"Creato Master Flat Normalizzato con mappa d'errore da {len(flat_files)} frame.")

print("\nRiduzione dati e propagazione degli errori completata!")

Processing Bias frames...
Creato Master Bias con mappa d'errore da 9 frame.

Processing Dark frames...
Creato Master Dark con mappa d'errore da 10 frame.

Processing Flat frames...
Creato Master Flat Normalizzato con mappa d'errore da 9 frame.

Riduzione dati e propagazione degli errori completata!


In [8]:
import os
import glob
import numpy as np
from astropy.io import fits

# Nome cartella contenente le immagini grezze (raw)
input_dir = "sci"  
# Nome cartella in cui verranno salvate le immagini ridotte (calibrate)
output_dir = "red" 

# ---------------------------------------------------------
# 1. CARICAMENTO DEI MASTER CALIBRATION (E RELATIVI ERRORI)
# ---------------------------------------------------------
# Master Bias
with fits.open("master_bias.fits") as h:
    master_bias = h[0].data.astype(np.float64)
with fits.open("master_bias_err.fits") as h:
    bias_err = h[0].data.astype(np.float64)

# Master Dark
with fits.open("master_dark.fits") as h:
    master_dark = h[0].data.astype(np.float64)
    dark_exptime = h[0].header.get("EXPTIME", 30)  # Tempo di esposizione del Master Dark
with fits.open("master_dark_err.fits") as h:
    dark_err = h[0].data.astype(np.float64)

# Master Flat
with fits.open("master_flat.fits") as h:
    master_flat = h[0].data.astype(np.float64)
with fits.open("master_flat_err.fits") as h:
    flat_err = h[0].data.astype(np.float64)

print("Master caricati con successo!\n")

# ---------------------------------------------------------
# 2. RICERCA DI TUTTI I FRAME GREZZI (RAW) DI AE UMa
# ---------------------------------------------------------
# Cerchiamo tutti i file .fit o .fits che iniziano con AE_UMa-
search_pattern = os.path.join(input_dir, "AE_UMa-*.fit*")
raw_files = glob.glob(search_pattern)

if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in '{input_dir}' con il pattern 'AE_UMa-*.fit*'.")

print(f"Trovati {len(raw_files)} file grezzi da elaborare.")

# ---------------------------------------------------------
# 3. LOOP DI RIDUZIONE ED ERRORE SU TUTTI I FILE
# ---------------------------------------------------------
for i, filepath in enumerate(sorted(raw_files), 1):
    filename = os.path.basename(filepath)
    print(f"[{i}/{len(raw_files)}] Riduzione in corso: {filename}...")
    
    # 3.1 Lettura del singolo frame raw
    with fits.open(filepath) as h_sci:
        sci_header = h_sci[0].header
        raw_data = h_sci[0].data.astype(np.float64)
        
        # Estrazione metadati con gestione dei default per keyword mancanti
        gain = sci_header["EGAIN"]
        rdnoise = sci_header.get("RDNOISE", 9.0)     # Default stimato di 9.0e-
        sci_exptime = sci_header["EXPTIME"] # Tempo di esposizione del singolo frame
        
    # 3.2 Ridimensionamento del Dark in base al tempo di esposizione del frame corrente
    dark_scale = sci_exptime / dark_exptime
    scaled_dark = master_dark * dark_scale
    scaled_dark_err = dark_err * dark_scale
    
    # 3.3 Riduzione matematica e conversione in elettroni
    numerator_adu = raw_data - master_bias - scaled_dark
    numerator_electrons = numerator_adu * gain
    
    # Divisione per il flat (con gestione degli errori di divisione per zero)
    with np.errstate(divide='ignore', invalid='ignore'):
        calibrated_science_electrons = numerator_electrons / master_flat
            
    # 3.4 Propagazione rigorosa dell'errore (in Elettroni)
    # Rumore di Poisson sui conteggi raw convertiti in elettroni
    shot_noise_electrons = np.sqrt(np.maximum(raw_data * gain, 0))
    
    # Combinazione quadratica degli errori al numeratore
    variance_numerator = (
        shot_noise_electrons**2 + 
        rdnoise**2 + 
        (gain * bias_err)**2 + 
        (gain * scaled_dark_err)**2
    )
    sigma_numerator = np.sqrt(variance_numerator)
    
    # Protezione del numeratore da valori nulli per il calcolo dell'errore relativo
    safe_numerator = np.where(numerator_electrons == 0, 1e-5, numerator_electrons)
    
    # Incertezza relativa e calcolo errore finale
    with np.errstate(divide='ignore', invalid='ignore'):
        relative_variance = (sigma_numerator / safe_numerator)**2 + (flat_err / master_flat)**2
        final_error_electrons = np.abs(calibrated_science_electrons) * np.sqrt(relative_variance)
        
    # Identifichiamo la maschera dei pixel non validi PRIMA di sovrascrivere nulla
    bad_pixel_mask = ~np.isfinite(calibrated_science_electrons) | ~np.isfinite(final_error_electrons)

    # Invece di azzerare, marchiamo esplicitamente come NaN
    calibrated_science_electrons[bad_pixel_mask] = np.nan
    final_error_electrons[bad_pixel_mask] = np.nan

    # Costruiamo una estensione di Data Quality (0 = buono, 1 = invalido/flat nullo)
    dq_array = bad_pixel_mask.astype(np.uint8)
    
    # 3.5 Creazione e salvataggio del file Multi-Extension FITS (MEF)
    # Prepariamo l'header originale aggiornando le informazioni dell'unità di misura
    primary_hdu = fits.PrimaryHDU(data=calibrated_science_electrons.astype(np.float32), header=sci_header)
    primary_hdu.header["OBJECT"] = "AE UMa"
    primary_hdu.header["BUNIT"] = "ELECTRONS"
    primary_hdu.header["EXTNAME"] = "SCI"
    primary_hdu.header["COMMENT"] = "Calibrato con pipeline di riduzione batch."
    dq_hdu = fits.ImageHDU(data=dq_array, name="DQ")
    dq_hdu.header["COMMENT"] = "0 = pixel valido, 1 = invalido (es. flat nullo)"
    
    # Estensione Errore
    err_hdu = fits.ImageHDU(data=final_error_electrons.astype(np.float32), name="ERR")
    err_hdu.header["BUNIT"] = "ELECTRONS"
    
    # Unione ed esportazione su disco
    mef_hdul = fits.HDUList([primary_hdu, err_hdu, dq_hdu])
    
    # Costruiamo il nuovo nome del file con il prefisso "reduced_"
    output_filename = f"reduced_{filename}"
    output_filepath = os.path.join(output_dir, output_filename)
    
    mef_hdul.writeto(output_filepath, overwrite=True)

print("\nProcesso di calibrazione batch completato con successo!")
print(f"Tutti i file ridotti sono salvati nella cartella '{output_dir}'.")

Master caricati con successo!

Trovati 314 file grezzi da elaborare.
[1/314] Riduzione in corso: AE_UMa-0001.fit...
[2/314] Riduzione in corso: AE_UMa-0002.fit...
[3/314] Riduzione in corso: AE_UMa-0003.fit...
[4/314] Riduzione in corso: AE_UMa-0004.fit...
[5/314] Riduzione in corso: AE_UMa-0005.fit...
[6/314] Riduzione in corso: AE_UMa-0006.fit...
[7/314] Riduzione in corso: AE_UMa-0007.fit...
[8/314] Riduzione in corso: AE_UMa-0008.fit...
[9/314] Riduzione in corso: AE_UMa-0009.fit...
[10/314] Riduzione in corso: AE_UMa-0010.fit...
[11/314] Riduzione in corso: AE_UMa-0011.fit...
[12/314] Riduzione in corso: AE_UMa-0012.fit...
[13/314] Riduzione in corso: AE_UMa-0013.fit...
[14/314] Riduzione in corso: AE_UMa-0014.fit...
[15/314] Riduzione in corso: AE_UMa-0015.fit...
[16/314] Riduzione in corso: AE_UMa-0016.fit...
[17/314] Riduzione in corso: AE_UMa-0017.fit...
[18/314] Riduzione in corso: AE_UMa-0018.fit...
[19/314] Riduzione in corso: AE_UMa-0019.fit...
[20/314] Riduzione in corso: